In [1]:
from nnsight import CONFIG

CONFIG.API.APIKEY = input("Enter your API key: ")
# print(CONFIG.API.APIKEY)

## list of remotely hosted models available here: https://nnsight.net/status/

# NDIF
NDIF (National Deep Inference Fabric) is a platform developed by the NSF in collaboration with Northeastern University for hosting frontier open-weight LLMs so US-based academic researchers can access them, even if they do not have local GPU resources. We'll be using NDIF to replicate some of the results in "Scaling Monosemanticity" on Eleuther AI's Pythia Language model. 

First, a brief tutorial on how to load a model on NDIF. We'll load deepseek, but there are many many other LLMs available. You can view the list here: https://nnsight.net/status/


In [2]:
from nnsight import LanguageModel
llm = LanguageModel(
    "deepseek-ai/DeepSeek-V3",
    device_map="auto",
    trust_remote_code=True
)
print(llm)

DeepseekV3ForCausalLM(
  (model): DeepseekV3Model(
    (embed_tokens): Embedding(129280, 7168)
    (layers): ModuleList(
      (0-2): 3 x DeepseekV3DecoderLayer(
        (self_attn): DeepseekV3Attention(
          (q_a_proj): Linear(in_features=7168, out_features=1536, bias=False)
          (q_a_layernorm): DeepseekV3RMSNorm()
          (q_b_proj): Linear(in_features=1536, out_features=24576, bias=False)
          (kv_a_proj_with_mqa): Linear(in_features=7168, out_features=576, bias=False)
          (kv_a_layernorm): DeepseekV3RMSNorm()
          (kv_b_proj): Linear(in_features=512, out_features=32768, bias=False)
          (o_proj): Linear(in_features=16384, out_features=7168, bias=False)
          (rotary_emb): DeepseekV3YarnRotaryEmbedding()
        )
        (mlp): DeepseekV3MLP(
          (gate_proj): Linear(in_features=7168, out_features=18432, bias=False)
          (up_proj): Linear(in_features=7168, out_features=18432, bias=False)
          (down_proj): Linear(in_features=18432

This is the deepseek model. There is a pretrained SAE available for Deepseek reasoning models trained by Goodfire (https://www.goodfire.ai/research/under-the-hood-of-a-reasoning-model#), but it is not compatible with NDIF. So we are going to use the pretrained SAE for Eleuther AI's Pythia 70M model

In [8]:
from nnsight import LanguageModel
from dictionary_learning.dictionary_learning.dictionary import AutoEncoder
import torch
from IPython.display import clear_output

In [ ]:
# Load pretrained autoencoder
# !./pretrained_dictionary_downloader.sh
# clear_output()

weights_path = "./dictionaries/pythia-70m-deduped/mlp_out_layer0/10_32768/ae.pt"
activation_dim = 512 # dimension of the NN's activations to be autoencoded
dictionary_size = 64 * activation_dim # number of features in the dictionary

ae = AutoEncoder(activation_dim, dictionary_size)
ae.load_state_dict(torch.load(weights_path,weights_only=True, map_location='cpu'))
# ae.cuda() # enable for cuda-capable systems


<All keys matched successfully>

# Applying SAE to Pythia70M

In [13]:
# now you should read through the scaling monosemanticity whitepaper and see if you can replicate some of hteir findings on another open source modle
model = LanguageModel("EleutherAI/pythia-70m-deduped", device_map="auto")
tokenizer = model.tokenizer

prompt = """
Call me Ishmael. Some years ago--never mind how long precisely--having little or no money in my purse, and nothing particular to interest me on shore, I thought I would sail about a little and see the watery part of the world. It is a way I have of driving off the spleen and regulating the circulation. Whenever I find myself growing grim about the mouth; whenever it is a damp, drizzly November in my soul; whenever I find myself involuntarily pausing before coffin warehouses, and bringing up the rear of every funeral I meet; and especially whenever my hypos get such an upper hand of me, that it requires a strong moral principle to prevent me from deliberately stepping into the street, and methodically knocking people's hats off--then, I account it high time to get to sea as soon as I can.
"""

# Extract layer 0 MLP output from base model
with model.trace(prompt) as tracer:
    mlp_0 = model.gpt_neox.layers[0].mlp.output.save()

# Use SAE to get features from activations
features = ae.encode(mlp_0)

config.json:   0%|          | 0.00/567 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/396 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/166M [00:00<?, ?B/s]

In [14]:
# Find top features using the autoencoder
summed_activations = features.abs().sum(dim=1) # Sort by max activations
top_activations_indices = summed_activations.topk(20).indices # Get indices of top 20

compounded = []
for i in top_activations_indices[0]:
    compounded.append(features[:,:,i.item()].cpu()[0])

compounded = torch.stack(compounded, dim=0)

# Visualizing Features

In [15]:

from circuitsvis.tokens import colored_tokens_multi

tokens = tokenizer.encode(prompt)
str_tokens = [tokenizer.decode(t) for t in tokens]

# Visualize activations for top 20 most prominent features
colored_tokens_multi(str_tokens, compounded.T)